In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 00 - Setup & Raw Data Generation
# MAGIC Generates raw orders + customers data for the pipeline.
# MAGIC Run this first, or as the first task in the workflow.

from pyspark.sql.functions import *

# Config — passed as widgets so workflow can override
dbutils.widgets.text("catalog", "delta_catalog")
dbutils.widgets.text("schema",  "delta_demo")
dbutils.widgets.text("num_orders",    "10000")
dbutils.widgets.text("num_customers", "200")

CATALOG       = dbutils.widgets.get("catalog")
SCHEMA        = dbutils.widgets.get("schema")
NUM_ORDERS    = int(dbutils.widgets.get("num_orders"))
NUM_CUSTOMERS = int(dbutils.widgets.get("num_customers"))

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# COMMAND ----------
# MAGIC %md ## Generate Raw Orders

orders_raw = spark.range(0, NUM_ORDERS) \
    .withColumnRenamed("id", "order_id") \
    .withColumn("customer_id", (col("order_id") % NUM_CUSTOMERS).cast("int")) \
    .withColumn("product_id",  (col("order_id") % 50).cast("int")) \
    .withColumn("category",
        when(col("product_id") < 15, "Electronics")
        .when(col("product_id") < 30, "Clothing")
        .otherwise("Grocery")
    ) \
    .withColumn("amount",     (rand() * 10000).cast("int")) \
    .withColumn("order_date", date_add(lit("2025-01-01"), 
                                       (col("order_id") % 120).cast("int"))) \
    .withColumn("status",
        when(col("order_id") % 10 == 0, "cancelled")
        .otherwise("completed")
    ) \
    .withColumn("year",  year("order_date")) \
    .withColumn("month", month("order_date"))

# Write to a landing table — this is what Bronze reads from
orders_raw.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.landing_orders")

# COMMAND ----------
# MAGIC %md ## Generate Raw Customers

customers_raw = spark.range(0, NUM_CUSTOMERS) \
    .withColumnRenamed("id", "customer_id") \
    .withColumn("name", concat(lit("Customer_"), col("customer_id"))) \
    .withColumn("city",
        when(col("customer_id") % 4 == 0, "Delhi")
        .when(col("customer_id") % 4 == 1, "Mumbai")
        .when(col("customer_id") % 4 == 2, "Bangalore")
        .otherwise("Chennai")
    ) \
    .withColumn("tier",
        when(col("customer_id") % 3 == 0, "Gold")
        .when(col("customer_id") % 3 == 1, "Silver")
        .otherwise("Bronze")
    ) \
    .withColumn("email", concat(lit("customer_"), col("customer_id"), 
                                lit("@email.com")))

customers_raw.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.landing_customers")

# COMMAND ----------
print(f"Setup complete:")
print(f"  Orders:    {spark.table(f'{CATALOG}.{SCHEMA}.landing_orders').count()}")
print(f"  Customers: {spark.table(f'{CATALOG}.{SCHEMA}.landing_customers').count()}")

dbutils.notebook.exit("00_setup: SUCCESS")